In [2]:

import nltk
from nltk.util import ngrams
from collections import defaultdict, Counter
import math
import random
import re

# Load and preprocess data
def load_data(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        text = file.read()
    sentences = text.split('\n')
    sentences = [re.sub(r"[^a-zA-Z0-9\s]", "", sentence.lower()) for sentence in sentences]
    sentences = [sentence.split() for sentence in sentences if sentence]
    return sentences

# Build the N-gram model
def build_ngram_model(sentences, n):
    ngram_freq = defaultdict(Counter)
    for sentence in sentences:
        sentence = ['<s>'] * (n - 1) + sentence + ['</s>']
        for ngram in ngrams(sentence, n):
            ngram_freq[ngram[:-1]][ngram[-1]] += 1
    return ngram_freq

# Estimate probabilities using Laplace smoothing
def estimate_probabilities(ngram_freq, vocab_size, alpha=1):
    probabilities = {}
    for context, words in ngram_freq.items():
        total_count = sum(words.values()) + alpha * vocab_size
        probabilities[context] = {word: (count + alpha) / total_count for word, count in words.items()}
    return probabilities

# Generate text
def generate_text(model, n, max_words=20, seed_words=None):
    if seed_words is None:
        seed_words = ['<s>'] * (n - 1)
    else:
        seed_words = ['<s>'] * (n - len(seed_words) - 1) + seed_words

    sentence = seed_words.copy()
    while len(sentence) < max_words + n - 1:
        context = tuple(sentence[-(n - 1):])
        next_word = random.choices(
            list(model.get(context, {'</s>': 1}).keys()),
            list(model.get(context, {'</s>': 1}).values())
        )[0]
        if next_word == '</s>':
            break
        sentence.append(next_word)

    return ' '.join(sentence[n-1:])

# Evaluate the model using Perplexity
def calculate_perplexity(model, sentences, n, vocab_size, alpha=1):
    log_prob_sum = 0
    token_count = 0

    for sentence in sentences:
        sentence = ['<s>'] * (n - 1) + sentence + ['</s>']
        for ngram in ngrams(sentence, n):
            context, word = ngram[:-1], ngram[-1]
            prob = model.get(context, {}).get(word, alpha / (sum(model.get(context, {}).values()) + alpha * vocab_size))
            log_prob_sum += math.log(prob)
            token_count += 1

    return math.exp(-log_prob_sum / token_count)

# Main execution
if __name__ == "__main__":
    # Filepath
    filepath = "./eng_news_2017_10K-sentences.txt"

    # Load and preprocess data
    sentences = load_data(filepath)

    # Vocabulary
    all_words = [word for sentence in sentences for word in sentence]
    vocab = set(all_words)
    vocab_size = len(vocab)

    # Build and test the N-gram model
    for n in range(1, 5):  # bigram to 4-gram
        print(f"\nN={n} Model:")
        ngram_freq = build_ngram_model(sentences, n)
        model = estimate_probabilities(ngram_freq, vocab_size)

        # Generate text
        print("Generated Text:", generate_text(model, n))

        # Evaluate model
        train_data = sentences[:int(len(sentences)*0.8)]
        test_data = sentences[int(len(sentences)*0.8):]
        perplexity = calculate_perplexity(model, test_data, n, vocab_size)
        print("Perplexity:", perplexity)



N=1 Model:
Generated Text: cure
Perplexity: 1825.0700921755795

N=2 Model:
Generated Text: 5069 much easier than tripled between forces firing neatly confirms that were considered by demonstrating its overnight
Perplexity: 7922.125773778329

N=3 Model:
Generated Text: 929 autumnfest entertainment chairman jeffrey gamache thinks duprey protests a bit too distraught about what happened johnston reflects on the
Perplexity: 15489.693271728664

N=4 Model:
Generated Text: 144 after 13 years on 947 breakfast radio darren simpson will move to the afternoon drive and anele mdoda will
Perplexity: 16790.43721528352
